In [3]:
import pandas as pd
import numpy as np 

In [4]:
link="https://en.wikipedia.org/wiki/List_of_American_films_of_2021"
link

'https://en.wikipedia.org/wiki/List_of_American_films_of_2021'

In [5]:
import requests
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_American_films_of_2021"  # replace with your link

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url, headers=headers)

# Parse the HTML tables
tables = pd.read_html(response.text, header=0)

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_54608\2632168118.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, header=0)


In [6]:
df1.head()

,Opening,Opening.1,Title,Company credits,Cast and crew,Ref.
0,J A N U A R Y,1,Shadow in the Cloud,Vertical Entertainment,Roseanne Liang (director/screenplay); Max Land...,[2]
1,J A N U A R Y,5,Hacksaw,Leone Films / Midnight Releasing,"Anthony Leone (director/screenplay); Amy Cay, ...",[3]
2,J A N U A R Y,12,Dr. Bird's Advice for Sad Poets,Relativity Media / Ketchup Entertainment,Yaniv Raz (director/screenplay); Lucas Jade Zu...,[4]
3,J A N U A R Y,13,The White Tiger,Netflix / ARRAY / Purple Pebble Pictures,Ramin Bahrani (director/screenplay); Adarsh Go...,NaN
4,J A N U A R Y,14,Locked Down,HBO Max / Warner Bros. Pictures,Doug Liman (director); Steven Knight (screenpl...,[5]


In [7]:
df3.tail()

,Opening,Opening.1,Title,Company credits,Cast and crew,Ref.
106,S E P T E M B E R,24,My Little Pony: A New Generation,Netflix / Entertainment One / Boulder Media,"Robert Cullen, Jose Ucha (directors); Gillian ...",[201]
107,S E P T E M B E R,24,Dear Evan Hansen,Universal Pictures / Perfect World Pictures,Stephen Chbosky (director); Steven Levenson (s...,[202]
108,S E P T E M B E R,24,The Guilty,Netflix / Bold Films / Nine Stories Productions,Antoine Fuqua (director); Nic Pizzolatto (scre...,[173]
109,S E P T E M B E R,24,Birds of Paradise,Amazon Studios,Sarah Adina Smith (director/screenplay); Krist...,[203]
110,S E P T E M B E R,30,After We Fell,Voltage Pictures,Castille Landon (director); Sharon Soboil (scr...,[204]


In [8]:
df=pd.concat([df1,df2,df3,df4],ignore_index=True)
df.head()

,Opening,Opening.1,Title,Company credits,Cast and crew,Ref.
0,J A N U A R Y,1,Shadow in the Cloud,Vertical Entertainment,Roseanne Liang (director/screenplay); Max Land...,[2]
1,J A N U A R Y,5,Hacksaw,Leone Films / Midnight Releasing,"Anthony Leone (director/screenplay); Amy Cay, ...",[3]
2,J A N U A R Y,12,Dr. Bird's Advice for Sad Poets,Relativity Media / Ketchup Entertainment,Yaniv Raz (director/screenplay); Lucas Jade Zu...,[4]
3,J A N U A R Y,13,The White Tiger,Netflix / ARRAY / Purple Pebble Pictures,Ramin Bahrani (director/screenplay); Adarsh Go...,NaN
4,J A N U A R Y,14,Locked Down,HBO Max / Warner Bros. Pictures,Doug Liman (director); Steven Knight (screenpl...,[5]


In [9]:
df.count()

Opening            368
Opening.1          367
Title              366
Company credits    366
Cast and crew      366
Ref.               302
dtype: int64

In [10]:
df.isnull().sum()

Opening             0
Opening.1           1
Title               2
Company credits     2
Cast and crew       2
Ref.               66
dtype: int64

In [11]:
from tmdbv3api import TMDb
import json
import requests
tmdb=TMDb()
tmdb.api_key='08d38c60a56fe3fdf55456105dab7193'

In [12]:


from tmdbv3api import Movie
tmdb_movie = Movie()
def get_genre(x): #pass in the title of the movies
    genres = []
    result = tmdb_movie.search(x) #the title will be searched in the tmdb_movie
    movie_id = result[0].id #we will match the "id" with the "title"
    response = requests.get('https://api.themoviedb.org/3/movie/{}?api_key={}'.format(movie_id,tmdb.api_key)) #we will get the result from the IMDb data
    data_json = response.json() #we will then convert it to a json file
    if data_json['genres']: #in the json file we will only need to extract the "genre"
        genre_str = " " 
        for i in range(0,len(data_json['genres'])):
            genres.append(data_json['genres'][i]['name']) #we will then add the "genre" to the empty genre list we created above
        return genre_str.join(genres)
    else:
        np.NaN # we will return the results but if we don't find anything we will consider it as a missing value


In [13]:
import time
from requests.exceptions import RequestException

# simple cache to avoid duplicate API calls for the same title
_genre_cache = {}

def get_genre_safe(title, max_retries=3, backoff=1.0, pause=0.25):
	# handle missing/empty titles
	if not title or title.lower() in ('nan', 'none', ''):
		return np.nan

	if title in _genre_cache:
		return _genre_cache[title]

	for attempt in range(max_retries):
		try:
			result = tmdb_movie.search(title)
			if not result:
				_genre_cache[title] = np.nan
				return np.nan

			movie_id = result[0].id
			response = requests.get(
				f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}',
				timeout=10
			)
			response.raise_for_status()
			data_json = response.json()

			genres = [g.get('name') for g in data_json.get('genres', []) if g.get('name')]
			value = ' '.join(genres) if genres else np.nan
			_genre_cache[title] = value

			# small pause to reduce likelihood of hitting rate limits / abrupt disconnects
			time.sleep(pause)
			return value

		except RequestException:
			# retry with exponential backoff on network errors
			if attempt < max_retries - 1:
				time.sleep(backoff * (2 ** attempt))
				continue
			_genre_cache[title] = np.nan
			return np.nan
		except Exception:
			# catch-all: return NaN for unexpected failures
			_genre_cache[title] = np.nan
			return np.nan

# apply the safe function to Title column
df['genres'] = df['Title'].fillna('').astype(str).map(lambda x: get_genre_safe(x))
df

KeyboardInterrupt: 

In [ ]:
df_2021=df[["Title","Cast and crew","genres"]]
df_2021.head()

,Title,Cast and crew,genres
0,Shadow in the Cloud,Roseanne Liang (director/screenplay); Max Land...,Horror Action War
1,Hacksaw,"Anthony Leone (director/screenplay); Amy Cay, ...",Adventure Family TV Movie Western
2,Dr. Bird's Advice for Sad Poets,Yaniv Raz (director/screenplay); Lucas Jade Zu...,NaN
3,The White Tiger,Ramin Bahrani (director/screenplay); Adarsh Go...,Drama
4,Locked Down,Doug Liman (director); Steven Knight (screenpl...,Comedy Crime Romance


In [ ]:
def get_director(x):
    if "(director)" in x:
        return x.split("(director)")[0]
    elif "(directors)" in x:
        return x.split("(directors)")[0]
    else:
        return x.split("(director/screenplay)")[0]
    

In [ ]:
df_2021["director_name"]=df_2021["Cast and crew"].map(lambda x:get_director(str(x)))
df_2021

C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\2742680864.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2021["director_name"]=df_2021["Cast and crew"].map(lambda x:get_director(str(x)))


,Title,Cast and crew,genres,director_name
0,Shadow in the Cloud,Roseanne Liang (director/screenplay); Max Land...,Horror Action War,Roseanne Liang
1,Hacksaw,"Anthony Leone (director/screenplay); Amy Cay, ...",Adventure Family TV Movie Western,Anthony Leone
2,Dr. Bird's Advice for Sad Poets,Yaniv Raz (director/screenplay); Lucas Jade Zu...,NaN,Yaniv Raz
3,The White Tiger,Ramin Bahrani (director/screenplay); Adarsh Go...,Drama,Ramin Bahrani
4,Locked Down,Doug Liman (director); Steven Knight (screenpl...,Comedy Crime Romance,Doug Liman
...,...,...,...,...
362,The Tragedy of Macbeth,Joel Coen (director/screenplay); Denzel Washin...,Drama Action Fantasy,Joel Coen
363,A Journal for Jordan,Denzel Washington (director); Virgil Williams ...,Drama Romance,Denzel Washington
364,American Underdog,"Erwin brothers (directors); Jon Erwin, David A...",Drama Family,Erwin brothers
365,Memoria,Apichatpong Weerasethakul (director/acreenplay...,NaN,Apichatpong Weerasethakul (director/acreenplay...


In [ ]:
def get_actor1(x):
    return ((x.split("screenplay);")[-1]).split(",")[0])


In [ ]:
df_2021["actor_1_name"]=df_2021["Cast and crew"].map(lambda x:get_actor1(str(x)))
df_2021

,movie_title,Cast and crew,genres,director_name,acotr_1_name,acotr_2_name,acotr_3_name,actor_1_name
0,Shadow in the Cloud,Roseanne Liang (director/screenplay); Max Land...,Horror Action War,Roseanne Liang,Chloë Grace Moretz,Taylor John Smith,13.0,Chloë Grace Moretz
1,Hacksaw,"Anthony Leone (director/screenplay); Amy Cay, ...",Adventure Family TV Movie Western,Anthony Leone,Amy Cay,Brian Patrick Butler,19.0,Amy Cay
2,Dr. Bird's Advice for Sad Poets,Yaniv Raz (director/screenplay); Lucas Jade Zu...,NaN,Yaniv Raz,Lucas Jade Zumann,Taylor Russell,13.0,Lucas Jade Zumann
3,The White Tiger,Ramin Bahrani (director/screenplay); Adarsh Go...,Drama,Ramin Bahrani,Adarsh Gourav,Rajkummar Rao,22.0,Adarsh Gourav
4,Locked Down,Doug Liman (director); Steven Knight (screenpl...,Comedy Crime Romance,Doug Liman,Anne Hathaway,Chiwetel Ejiofor,17.0,Anne Hathaway
...,...,...,...,...,...,...,...,...
362,The Tragedy of Macbeth,Joel Coen (director/screenplay); Denzel Washin...,Drama Action Fantasy,Joel Coen,Denzel Washington,Frances McDormand,14.0,Denzel Washington
363,A Journal for Jordan,Denzel Washington (director); Virgil Williams ...,Drama Romance,Denzel Washington,Michael B. Jordan,Chanté Adams,16.0,Michael B. Jordan
364,American Underdog,"Erwin brothers (directors); Jon Erwin, David A...",Drama Family,Erwin brothers,Zachary Levi,Anna Paquin,13.0,Zachary Levi
365,Memoria,Apichatpong Weerasethakul (director/acreenplay...,NaN,Apichatpong Weerasethakul (director/acreenplay...,Apichatpong Weerasethakul (director/acreenplay...,Elkin Díaz,15.0,Apichatpong Weerasethakul (director/acreenplay...


In [ ]:
def get_actor2(x):
    if len((x.split("screenplay);")[-1]).split(","))<2:
        return np.nan 
    else:
        return ((x.split("screenplay);")[-1]).split(",")[1])

In [ ]:
df_2021["actor_2_name"]=df_2021["Cast and crew"].map(lambda x:get_actor2(str(x)))
df_2021.head()

,movie_title,Cast and crew,genres,director_name,acotr_1_name,acotr_2_name,acotr_3_name,actor_1_name,actor_2_name
0,Shadow in the Cloud,Roseanne Liang (director/screenplay); Max Land...,Horror Action War,Roseanne Liang,Chloë Grace Moretz,Taylor John Smith,13.0,Chloë Grace Moretz,Taylor John Smith
1,Hacksaw,"Anthony Leone (director/screenplay); Amy Cay, ...",Adventure Family TV Movie Western,Anthony Leone,Amy Cay,Brian Patrick Butler,19.0,Amy Cay,Brian Patrick Butler
2,Dr. Bird's Advice for Sad Poets,Yaniv Raz (director/screenplay); Lucas Jade Zu...,NaN,Yaniv Raz,Lucas Jade Zumann,Taylor Russell,13.0,Lucas Jade Zumann,Taylor Russell
3,The White Tiger,Ramin Bahrani (director/screenplay); Adarsh Go...,Drama,Ramin Bahrani,Adarsh Gourav,Rajkummar Rao,22.0,Adarsh Gourav,Rajkummar Rao
4,Locked Down,Doug Liman (director); Steven Knight (screenpl...,Comedy Crime Romance,Doug Liman,Anne Hathaway,Chiwetel Ejiofor,17.0,Anne Hathaway,Chiwetel Ejiofor


In [ ]:
def actor3(x):
    if len((x.split("screenplay);")[-1]).split(","))<3:
        return np.nan
    else:
        return len((x.split("screenplay);")[-1]).split(",")[2])
    

In [ ]:
df_2021["actor_3_name"]=df["Cast and crew"].map(lambda x:actor3(str(x)))

In [ ]:
df_2021.head()

,Title,Cast and crew,genres,director_name,acotr_1_name,acotr_2_name,acotr_3_name
0,Shadow in the Cloud,Roseanne Liang (director/screenplay); Max Land...,Horror Action War,Roseanne Liang,Chloë Grace Moretz,Taylor John Smith,13.0
1,Hacksaw,"Anthony Leone (director/screenplay); Amy Cay, ...",Adventure Family TV Movie Western,Anthony Leone,Amy Cay,Brian Patrick Butler,19.0
2,Dr. Bird's Advice for Sad Poets,Yaniv Raz (director/screenplay); Lucas Jade Zu...,NaN,Yaniv Raz,Lucas Jade Zumann,Taylor Russell,13.0
3,The White Tiger,Ramin Bahrani (director/screenplay); Adarsh Go...,Drama,Ramin Bahrani,Adarsh Gourav,Rajkummar Rao,22.0
4,Locked Down,Doug Liman (director); Steven Knight (screenpl...,Comedy Crime Romance,Doug Liman,Anne Hathaway,Chiwetel Ejiofor,17.0


In [ ]:
df_2021 = df_2021.rename(columns={"Title":"movie_title"})

In [ ]:
new_df_21=df_2021.loc[:,["director_name","actor_1_name","actor_2_name","actor_3_name","genres","movie_title"]]

In [ ]:
new_df_21.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Roseanne Liang,Chloë Grace Moretz,Taylor John Smith,13.0,Horror Action War,Shadow in the Cloud
1,Anthony Leone,Amy Cay,Brian Patrick Butler,19.0,Adventure Family TV Movie Western,Hacksaw
2,Yaniv Raz,Lucas Jade Zumann,Taylor Russell,13.0,NaN,Dr. Bird's Advice for Sad Poets
3,Ramin Bahrani,Adarsh Gourav,Rajkummar Rao,22.0,Drama,The White Tiger
4,Doug Liman,Anne Hathaway,Chiwetel Ejiofor,17.0,Comedy Crime Romance,Locked Down


In [ ]:
new_df_21["actor_1_name"]=new_df_21["actor_1_name"].replace("Unknown","unknown")

In [ ]:
new_df_21["actor_2_name"]=new_df_21["actor_2_name"].replace(np.nan,"unknown")


In [ ]:
new_df_21["actor_3_name"]=new_df_21["actor_3_name"].replace(np.nan,"unknown")

In [ ]:
new_df_21["movie_title"]=new_df_21["movie_title"].str.lower()

In [ ]:
new_df_21['comb'] = (new_df_21['actor_1_name'].astype(str) + ' ' +
                     new_df_21['actor_2_name'].astype(str) + ' ' +
                     new_df_21['actor_3_name'].astype(str) + ' ' +
                     new_df_21['director_name'].astype(str) + ' ' +
                     new_df_21['genres'].astype(str))


In [ ]:
new_df_21.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Roseanne Liang,Chloë Grace Moretz,Taylor John Smith,13.0,Horror Action War,shadow in the cloud,Chloë Grace Moretz Taylor John Smith 13.0 Ro...
1,Anthony Leone,Amy Cay,Brian Patrick Butler,19.0,Adventure Family TV Movie Western,hacksaw,Amy Cay Brian Patrick Butler 19.0 Anthony Le...
2,Yaniv Raz,Lucas Jade Zumann,Taylor Russell,13.0,NaN,dr. bird's advice for sad poets,Lucas Jade Zumann Taylor Russell 13.0 Yaniv ...
3,Ramin Bahrani,Adarsh Gourav,Rajkummar Rao,22.0,Drama,the white tiger,Adarsh Gourav Rajkummar Rao 22.0 Ramin Bahra...
4,Doug Liman,Anne Hathaway,Chiwetel Ejiofor,17.0,Comedy Crime Romance,locked down,Anne Hathaway Chiwetel Ejiofor 17.0 Doug Lim...


In [ ]:
link="https://en.wikipedia.org/wiki/List_of_American_films_of_2022"

In [ ]:
import requests
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_American_films_of_2022"  # replace with your link

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url, headers=headers)

# Parse the HTML tables
tables = pd.read_html(response.text, header=0)

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\2223701385.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, header=0)


In [ ]:
df1.head()

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,7,The 355,Universal Pictures / Freckle Films / FilmNatio...,Simon Kinberg (director/screenplay); Theresa R...,[2]
1,J A N U A R Y,7,The Legend of La Llorona,Saban Films / Ageless Pictures,Patricia Harris Seeley (director); José Prende...,[3]
2,J A N U A R Y,7,The Commando,Saban Films / Premiere Entertainment,Asif Akbar (director); Koji Steven Sakai (scre...,[4]
3,J A N U A R Y,7,American Siege,Vertical Entertainment,Edward John Drake (director/screenplay); Timot...,[5]
4,J A N U A R Y,14,Scream,Paramount Pictures / Spyglass Media Group / Ra...,"Matt Bettinelli-Olpin, Tyler Gillett (director...",[6]


In [ ]:
df=pd.concat([df1,df2,df3,df4],ignore_index=True)
df.head()

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,7.0,The 355,Universal Pictures / Freckle Films / FilmNatio...,Simon Kinberg (director/screenplay); Theresa R...,[2]
1,J A N U A R Y,7.0,The Legend of La Llorona,Saban Films / Ageless Pictures,Patricia Harris Seeley (director); José Prende...,[3]
2,J A N U A R Y,7.0,The Commando,Saban Films / Premiere Entertainment,Asif Akbar (director); Koji Steven Sakai (scre...,[4]
3,J A N U A R Y,7.0,American Siege,Vertical Entertainment,Edward John Drake (director/screenplay); Timot...,[5]
4,J A N U A R Y,14.0,Scream,Paramount Pictures / Spyglass Media Group / Ra...,"Matt Bettinelli-Olpin, Tyler Gillett (director...",[6]


In [ ]:
import time
from requests.exceptions import RequestException

# simple cache to avoid duplicate API calls for the same title
_genre_cache = {}

def get_genre_safe(title, max_retries=3, backoff=1.0, pause=0.25):
	# handle missing/empty titles
	if not title or title.lower() in ('nan', 'none', ''):
		return np.nan

	if title in _genre_cache:
		return _genre_cache[title]

	for attempt in range(max_retries):
		try:
			result = tmdb_movie.search(title)
			if not result:
				_genre_cache[title] = np.nan
				return np.nan

			movie_id = result[0].id
			response = requests.get(
				f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}',
				timeout=10
			)
			response.raise_for_status()
			data_json = response.json()

			genres = [g.get('name') for g in data_json.get('genres', []) if g.get('name')]
			value = ' '.join(genres) if genres else np.nan
			_genre_cache[title] = value

			# small pause to reduce likelihood of hitting rate limits / abrupt disconnects
			time.sleep(pause)
			return value

		except RequestException:
			# retry with exponential backoff on network errors
			if attempt < max_retries - 1:
				time.sleep(backoff * (2 ** attempt))
				continue
			_genre_cache[title] = np.nan
			return np.nan
		except Exception:
			# catch-all: return NaN for unexpected failures
			_genre_cache[title] = np.nan
			return np.nan

# apply the safe function to Title column
df['genres'] = df['Title'].fillna('').astype(str).map(lambda x: get_genre_safe(x))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres
0,J A N U A R Y,7.0,The 355,Universal Pictures / Freckle Films / FilmNatio...,Simon Kinberg (director/screenplay); Theresa R...,[2],Action Adventure Thriller
1,J A N U A R Y,7.0,The Legend of La Llorona,Saban Films / Ageless Pictures,Patricia Harris Seeley (director); José Prende...,[3],Horror Thriller
2,J A N U A R Y,7.0,The Commando,Saban Films / Premiere Entertainment,Asif Akbar (director); Koji Steven Sakai (scre...,[4],Action Crime Thriller
3,J A N U A R Y,7.0,American Siege,Vertical Entertainment,Edward John Drake (director/screenplay); Timot...,[5],NaN
4,J A N U A R Y,14.0,Scream,Paramount Pictures / Spyglass Media Group / Ra...,"Matt Bettinelli-Olpin, Tyler Gillett (director...",[6],Crime Horror Mystery
...,...,...,...,...,...,...,...
319,D E C E M B E R,30.0,"Alice, Darling",Lionsgate / Elevation Pictures,Mary Nighy (director); Alanna Francis (screenp...,[268],Thriller Drama Romance
320,D E C E M B E R,NaN,NaN,NaN,NaN,NaN,NaN
321,D E C E M B E R,NaN,NaN,NaN,NaN,NaN,NaN
322,D E C E M B E R,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_2022=df[["Title","Cast and crew","genres"]]
df_2022.head()

,Title,Cast and crew,genres
0,The 355,Simon Kinberg (director/screenplay); Theresa R...,Action Adventure Thriller
1,The Legend of La Llorona,Patricia Harris Seeley (director); José Prende...,Horror Thriller
2,The Commando,Asif Akbar (director); Koji Steven Sakai (scre...,Action Crime Thriller
3,American Siege,Edward John Drake (director/screenplay); Timot...,NaN
4,Scream,"Matt Bettinelli-Olpin, Tyler Gillett (director...",Crime Horror Mystery


In [ ]:
def get_director(x):
    if "(director)" in x:
        return x.split("(director)")[0]
    elif "(directors)" in x:
        return x.split("(directors)")[0]
    else:
        return x.split('(director/screenplay)')[0]
    

In [ ]:
df_2022["director_name"]=df_2022["Cast and crew"].map(lambda x:get_director(str(x)))


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\1855171887.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2022["director_name"]=df_2022["Cast and crew"].map(lambda x:get_director(str(x)))


In [ ]:
def get_actor1(x):
    return ((x.split("screenplay);")[-1]).split(",")[0])

In [ ]:
df_2022["actor_1_name"]=df_2022["Cast and crew"].map(lambda x:get_actor1(str(x)))


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\247615566.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2022["actor_1_name"]=df_2022["Cast and crew"].map(lambda x:get_actor1(str(x)))


In [ ]:
def get_actor2(x):
    if len((x.split("screenplay);")[-1]).split(','))<2:
        return np.nan
    else:
        return ((x.split("screenplay);")[-1]).split(',')[1])
    
        

In [ ]:
df_2022['actor_2_name']=df["Cast and crew"].map(lambda x:get_actor2(str(x)))

C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\252217291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2022['actor_2_name']=df["Cast and crew"].map(lambda x:get_actor2(str(x)))


In [ ]:
def get_actor3(x):
    if len((x.split("screenplay);")[-1]).split(","))<3:
        return np.nan
    else:
        return ((x.split("screenplay);")[-1]).split(",")[2])
    

In [ ]:
df_2022["actor_3_name"]=df['Cast and crew'].map(lambda x:get_actor3(str(x)))
df_2022.head()

,Title,Cast and crew,genres,director_name,actor_1_name,actor_2_name,actor_3_name
0,The 355,Simon Kinberg (director/screenplay); Theresa R...,Action Adventure Thriller,Simon Kinberg,Jessica Chastain,Penélope Cruz,Fan Bingbing
1,The Legend of La Llorona,Patricia Harris Seeley (director); José Prende...,Horror Thriller,Patricia Harris Seeley,Autumn Reeser,Danny Trejo,Antonio Cupo
2,The Commando,Asif Akbar (director); Koji Steven Sakai (scre...,Action Crime Thriller,Asif Akbar,Mickey Rourke,Michael Jai White,NaN
3,American Siege,Edward John Drake (director/screenplay); Timot...,NaN,Edward John Drake,Timothy V. Murphy,Bruce Willis,Rob Gough
4,Scream,"Matt Bettinelli-Olpin, Tyler Gillett (director...",Crime Horror Mystery,"Matt Bettinelli-Olpin, Tyler Gillett",Melissa Barrera,Mason Gooding,Jenna Ortega


In [ ]:
df_2022=df_2022.rename(columns={"Title":"movie_title"})

In [ ]:
new_df22=df_2022.loc[:,["director_name",'actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

new_df22.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Simon Kinberg,Jessica Chastain,Penélope Cruz,Fan Bingbing,Action Adventure Thriller,The 355
1,Patricia Harris Seeley,Autumn Reeser,Danny Trejo,Antonio Cupo,Horror Thriller,The Legend of La Llorona
2,Asif Akbar,Mickey Rourke,Michael Jai White,NaN,Action Crime Thriller,The Commando
3,Edward John Drake,Timothy V. Murphy,Bruce Willis,Rob Gough,NaN,American Siege
4,"Matt Bettinelli-Olpin, Tyler Gillett",Melissa Barrera,Mason Gooding,Jenna Ortega,Crime Horror Mystery,Scream


In [ ]:
new_df22.isna().sum()

director_name     0
actor_1_name      0
actor_2_name     11
actor_3_name     35
genres           74
movie_title       4
dtype: int64

In [ ]:
new_df22['actor_2_name']=new_df22['actor_2_name'].replace(np.nan,"unknown")


In [ ]:
new_df22["actor_3_name"]=new_df22["actor_3_name"].replace(np.nan,"unknown")


In [ ]:
new_df22["movie_title"]=new_df22["movie_title"].str.lower()

In [ ]:
new_df22["comb"]=new_df22["actor_1_name"].astype(str)+""+new_df22["actor_2_name"].astype(str)+""+new_df22["actor_3_name"].astype(str)+""+new_df22["director_name"]+""+new_df22['genres'].astype(str)

In [ ]:
new_df22.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Simon Kinberg,Jessica Chastain,Penélope Cruz,Fan Bingbing,Action Adventure Thriller,the 355,Jessica Chastain Penélope Cruz Fan BingbingSi...
1,Patricia Harris Seeley,Autumn Reeser,Danny Trejo,Antonio Cupo,Horror Thriller,the legend of la llorona,Autumn Reeser Danny Trejo Antonio CupoPatrici...
2,Asif Akbar,Mickey Rourke,Michael Jai White,unknown,Action Crime Thriller,the commando,Mickey Rourke Michael Jai WhiteunknownAsif Ak...
3,Edward John Drake,Timothy V. Murphy,Bruce Willis,Rob Gough,NaN,american siege,Timothy V. Murphy Bruce Willis Rob GoughEdwar...
4,"Matt Bettinelli-Olpin, Tyler Gillett",Melissa Barrera,Mason Gooding,Jenna Ortega,Crime Horror Mystery,scream,Melissa Barrera Mason Gooding Jenna OrtegaMat...


In [ ]:
link="https://en.wikipedia.org/wiki/List_of_American_films_of_2023"

In [ ]:
import requests
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_American_films_of_2023"  # replace with your link

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url, headers=headers)

# Parse the HTML tables
tables = pd.read_html(response.text, header=0)

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\2675460135.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, header=0)


In [ ]:
df1.head()

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,6,M3GAN,Universal Pictures / Blumhouse Productions / A...,Gerard Johnstone (director); Akela Cooper (scr...,[3]
1,J A N U A R Y,6,The Old Way,Saban Films / Saturn Films,Brett Donowho (director); Carl W. Lucas (scree...,[4]
2,J A N U A R Y,11,The Devil Conspiracy,Samuel Goldwyn Films,Nathan Frankowski (director); Ed Alan (screenp...,[5]
3,J A N U A R Y,13,Plane,Lionsgate / MadRiver Pictures / Di Bonaventura...,Jean-François Richet (director); Charles Cummi...,[6]
4,J A N U A R Y,13,House Party,Warner Bros. Pictures / New Line Cinema,"Calmatic (director); Jamal Olori, Stephen Glov...",[7]


In [ ]:
df=pd.concat([df1,df2,df3,df4],ignore_index=True)
df.head()


,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,6,M3GAN,Universal Pictures / Blumhouse Productions / A...,Gerard Johnstone (director); Akela Cooper (scr...,[3]
1,J A N U A R Y,6,The Old Way,Saban Films / Saturn Films,Brett Donowho (director); Carl W. Lucas (scree...,[4]
2,J A N U A R Y,11,The Devil Conspiracy,Samuel Goldwyn Films,Nathan Frankowski (director); Ed Alan (screenp...,[5]
3,J A N U A R Y,13,Plane,Lionsgate / MadRiver Pictures / Di Bonaventura...,Jean-François Richet (director); Charles Cummi...,[6]
4,J A N U A R Y,13,House Party,Warner Bros. Pictures / New Line Cinema,"Calmatic (director); Jamal Olori, Stephen Glov...",[7]


In [ ]:
import time
from requests.exceptions import RequestException

# simple cache to avoid duplicate API calls for the same title
_genre_cache = {}

def get_genre_safe(title, max_retries=3, backoff=1.0, pause=0.25):
	# handle missing/empty titles
	if not title or title.lower() in ('nan', 'none', ''):
		return np.nan

	if title in _genre_cache:
		return _genre_cache[title]

	for attempt in range(max_retries):
		try:
			result = tmdb_movie.search(title)
			if not result:
				_genre_cache[title] = np.nan
				return np.nan

			movie_id = result[0].id
			response = requests.get(
				f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}',
				timeout=10
			)
			response.raise_for_status()
			data_json = response.json()

			genres = [g.get('name') for g in data_json.get('genres', []) if g.get('name')]
			value = ' '.join(genres) if genres else np.nan
			_genre_cache[title] = value

			# small pause to reduce likelihood of hitting rate limits / abrupt disconnects
			time.sleep(pause)
			return value

		except RequestException:
			# retry with exponential backoff on network errors
			if attempt < max_retries - 1:
				time.sleep(backoff * (2 ** attempt))
				continue
			_genre_cache[title] = np.nan
			return np.nan
		except Exception:
			# catch-all: return NaN for unexpected failures
			_genre_cache[title] = np.nan
			return np.nan

# apply the safe function to Title column
df['genres'] = df['Title'].fillna('').astype(str).map(lambda x: get_genre_safe(x))
df.head()

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres
0,J A N U A R Y,6,M3GAN,Universal Pictures / Blumhouse Productions / A...,Gerard Johnstone (director); Akela Cooper (scr...,[3],Science Fiction Horror
1,J A N U A R Y,6,The Old Way,Saban Films / Saturn Films,Brett Donowho (director); Carl W. Lucas (scree...,[4],Western Drama
2,J A N U A R Y,11,The Devil Conspiracy,Samuel Goldwyn Films,Nathan Frankowski (director); Ed Alan (screenp...,[5],Horror Fantasy Science Fiction Thriller
3,J A N U A R Y,13,Plane,Lionsgate / MadRiver Pictures / Di Bonaventura...,Jean-François Richet (director); Charles Cummi...,[6],Action Adventure Thriller
4,J A N U A R Y,13,House Party,Warner Bros. Pictures / New Line Cinema,"Calmatic (director); Jamal Olori, Stephen Glov...",[7],Comedy


In [ ]:
df_2023=df[["Title","Cast and crew","genres"]]
df_2023.sample(5)

,Title,Cast and crew,genres
60,John Wick: Chapter 4,"Chad Stahelski (director); Shay Hatten, Michae...",Action Thriller Crime
229,Flora and Son,"John Carney (director/screenplay); Eve Hewson,...",Music Comedy Drama
64,Adalynn,Jacob Byrd (director); Jerrod D. Brito (screen...,Horror Mystery Drama
274,Fingernails,Christos Nikou (director/screenplay); Sam Stei...,Romance Science Fiction Drama
29,Somebody I Used to Know,Dave Franco (director/screenplay); Alison Brie...,Romance Comedy Drama


In [ ]:
df_2023["director_name"]=df_2023['Cast and crew'].map(lambda x:get_director(str(x)))


In [ ]:
df_2023["actor_1_name"]=df_2023["Cast and crew"].map(lambda x:get_actor1(str(x)))

C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\1102048035.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2023["actor_1_name"]=df_2023["Cast and crew"].map(lambda x:get_actor1(str(x)))


In [ ]:
df_2023["actor_2_name"]=df_2023["Cast and crew"].map(lambda x:get_actor2(str(x)))

C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\533605592.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2023["actor_2_name"]=df_2023["Cast and crew"].map(lambda x:get_actor2(str(x)))


In [ ]:
df_2023["actor_3_name"]=df_2023["Cast and crew"].map(lambda x:get_actor3(str(x)))

C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\2861025802.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2023["actor_3_name"]=df_2023["Cast and crew"].map(lambda x:get_actor3(str(x)))


In [ ]:
df_2023=df_2023.rename(columns={"Title":"movie_title"})

In [ ]:
df_2023.sample(5)

,movie_title,Cast and crew,genres,actor_1_name,actor_2_name,actor_3_name,director_name
183,Shortcomings,Randall Park (director); Adrian Tomine (screen...,Comedy Drama Romance,Justin H. Min,Sherry Cola,Ally Maki,Randall Park
228,Cassandro,Roger Ross Williams (director/screenplay); Dav...,NaN,Gael García Bernal,Roberta Colindrez,Perla De La Rosa,Roger Ross Williams
21,80 for Brady,Kyle Marvin (director/screenplay); Emily Halpe...,Comedy Drama,Lily Tomlin,Jane Fonda,Rita Moreno,Kyle Marvin
106,Guardians of the Galaxy Vol. 3,"James Gunn (director/screenplay); Chris Pratt,...",NaN,Chris Pratt,Zoe Saldaña,Dave Bautista,James Gunn
242,The Exorcist: Believer,David Gordon Green (director/screenplay); Pete...,Horror Thriller,Leslie Odom Jr.,Lidya Jewett,Olivia O’Neill,David Gordon Green


In [ ]:
new_df23=df_2023.loc[:,["director_name",'actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

new_df23.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Gerard Johnstone,Allison Williams,Violet McGraw,Amie Donald,Science Fiction Horror,M3GAN
1,Brett Donowho,Nicolas Cage,Ryan Kiera Armstrong,NaN,Western Drama,The Old Way
2,Nathan Frankowski,Alice Orr-Ewing,Joe Doyle,Eveline Hall,Horror Fantasy Science Fiction Thriller,The Devil Conspiracy
3,Jean-François Richet,Gerard Butler,Mike Colter,Yoson An,Action Adventure Thriller,Plane
4,Calmatic,Tosin Cole,Jacob Latimore,Karen Obilom,Comedy,House Party


In [ ]:
new_df23["actor_2_name"]=new_df23['actor_2_name'].replace(np.nan,"unknown")


In [ ]:
new_df23["actor_3_name"]=new_df23["actor_3_name"].replace(np.nan,"unknown")

In [ ]:
new_df23=new_df23.rename(columns={"Title":"movie_name"})

In [ ]:
new_df23["comb"]=new_df23["actor_1_name"].astype(str)+''+new_df23["actor_2_name"].astype(str)+""+new_df23["actor_3_name"].astype(str)+""+new_df23["director_name"].astype(str)+""+new_df23["genres"]

In [ ]:
new_df23.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Gerard Johnstone,Allison Williams,Violet McGraw,Amie Donald,Science Fiction Horror,M3GAN,Allison Williams Violet McGraw Amie DonaldGer...
1,Brett Donowho,Nicolas Cage,Ryan Kiera Armstrong,unknown,Western Drama,The Old Way,Nicolas Cage Ryan Kiera ArmstrongunknownBrett...
2,Nathan Frankowski,Alice Orr-Ewing,Joe Doyle,Eveline Hall,Horror Fantasy Science Fiction Thriller,The Devil Conspiracy,Alice Orr-Ewing Joe Doyle Eveline HallNathan ...
3,Jean-François Richet,Gerard Butler,Mike Colter,Yoson An,Action Adventure Thriller,Plane,Gerard Butler Mike Colter Yoson AnJean-Franço...
4,Calmatic,Tosin Cole,Jacob Latimore,Karen Obilom,Comedy,House Party,Tosin Cole Jacob Latimore Karen ObilomCalmati...


2024 movies list


In [ ]:
import requests
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_American_films_of_2024"  # replace with your link

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url, headers=headers)

# Parse the HTML tables
tables = pd.read_html(response.text, header=0)

df1 = tables[2]
df2 = tables[3]
df3 = tables[4]
df4 = tables[5]


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\2286827388.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, header=0)


In [ ]:
df=pd.concat([df1,df2,df3,df4],ignore_index=True)
df.tail()

,Opening,Opening.1,Title,Company credits,Cast and crew,Ref.
490,D E C E M B E R,25.0,A Complete Unknown,Searchlight Pictures / Veritas Entertainment /...,James Mangold (director/screenplay); Jay Cocks...,[464]
491,D E C E M B E R,25.0,The Fire Inside,Metro-Goldwyn-Mayer,Rachel Morrison (director); Barry Jenkins (scr...,[465]
492,D E C E M B E R,25.0,Babygirl,A24 / 2AM,Halina Reijn (director/screenplay); Nicole Kid...,[466]
493,D E C E M B E R,25.0,Los Frikis,Wayward/Range / Lord Miller,"Michael Schwartz, Tyler Nilson (directors/scre...",[467]
494,D E C E M B E R,27.0,Bloody Axe Wound,Shudder / RLJE Films,Matthew John Lawrence (director/screenplay); M...,[468]


In [ ]:
import time
from requests.exceptions import RequestException

# simple cache to avoid duplicate API calls for the same title
_genre_cache = {}

def get_genre_safe(title, max_retries=3, backoff=1.0, pause=0.25):
	# handle missing/empty titles
	if not title or title.lower() in ('nan', 'none', ''):
		return np.nan

	if title in _genre_cache:
		return _genre_cache[title]

	for attempt in range(max_retries):
		try:
			result = tmdb_movie.search(title)
			if not result:
				_genre_cache[title] = np.nan
				return np.nan

			movie_id = result[0].id
			response = requests.get(
				f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}',
				timeout=10
			)
			response.raise_for_status()
			data_json = response.json()

			genres = [g.get('name') for g in data_json.get('genres', []) if g.get('name')]
			value = ' '.join(genres) if genres else np.nan
			_genre_cache[title] = value

			# small pause to reduce likelihood of hitting rate limits / abrupt disconnects
			time.sleep(pause)
			return value

		except RequestException:
			# retry with exponential backoff on network errors
			if attempt < max_retries - 1:
				time.sleep(backoff * (2 ** attempt))
				continue
			_genre_cache[title] = np.nan
			return np.nan
		except Exception:
			# catch-all: return NaN for unexpected failures
			_genre_cache[title] = np.nan
			return np.nan

# apply the safe function to Title column
df['genres'] = df['Title'].fillna('').astype(str).map(lambda x: get_genre_safe(x))
df.head()

,Opening,Opening.1,Title,Company credits,Cast and crew,Ref.,genres
0,J A N U A R Y,2.0,The Mummy Murders,Gravitas Ventures,Colin Bressler (director/screenplay); Will Don...,[3],Horror Crime
1,J A N U A R Y,3.0,Self Reliance,Neon / Hulu / MRC / Paramount Global Content D...,Jake Johnson (director/screenplay); Jake Johns...,[4],Comedy Thriller
2,J A N U A R Y,4.0,DarkGame,Gravitas Ventures,"Howard J. Ford (director); Gary Grant, Niall J...",[5],Horror Thriller
3,J A N U A R Y,5.0,Night Swim,Universal Pictures / Blumhouse Productions / A...,Bryce McGuire (director/screenplay); Wyatt Rus...,[6],NaN
4,J A N U A R Y,5.0,He Went That Way,Vertical Entertainment / Mister Smith Entertai...,Jeffrey Darling (director); Evan M. Wiener (sc...,[7],Thriller Crime Drama


In [ ]:
df_2024=df[["Title","Cast and crew","genres"]]
df_2024.sample(4)

,Title,Cast and crew,genres
151,Bloodline Killer,Ante Novakovic (director); Anthony and James G...,Horror Thriller
184,Sight,Andrew Hyatt (director/screenplay); John Duiga...,NaN
242,MaXXXine,"Ti West (director/screenplay); Mia Goth, Eliza...",NaN
149,The Feeling That the Time for Doing Something ...,Joanna Arnow (director/screenplay); Scott Cohe...,Comedy


In [ ]:
df_2024["director_name"]=df_2024["Cast and crew"].map(lambda x:get_director(str(x)))

C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\1225840667.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2024["director_name"]=df_2024["Cast and crew"].map(lambda x:get_director(str(x)))


In [ ]:
df_2024["actor_1_name"]=df_2024["Cast and crew"].map(lambda x:get_actor1(str(x)))


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\1801541037.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2024["actor_1_name"]=df_2024["Cast and crew"].map(lambda x:get_actor1(str(x)))


In [ ]:
df_2024["actor_2_name"]=df_2024["Cast and crew"].map(lambda x:get_actor2(str(x)))

C:\Users\Vishnu\AppData\Local\Temp\ipykernel_21972\2179488126.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2024["actor_2_name"]=df_2024["Cast and crew"].map(lambda x:get_actor2(str(x)))


In [ ]:
df_2024["actor_3_name"]=df_2024["Cast and crew"].map(lambda x:get_actor3(str(x)))

In [ ]:
df_2024=df_2024.rename(columns={"Title":"movie_title"})

In [ ]:
new_df24=df_2024.loc[:,["director_name","actor_1_name","actor_2_name","actor_3_name","genres","movie_title"]]

In [ ]:
new_df24.head()


,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Colin Bressler,Leila Annastasia Scott,Jason Scarbrough,Jeff Caperton,Horror Crime,The Mummy Murders
1,Jake Johnson,Jake Johnson,Anna Kendrick,Natalie Morales,Comedy Thriller,Self Reliance
2,Howard J. Ford,Ed Westwick,Andrew P. Stephen,Natalya Tsvetkova,Horror Thriller,DarkGame
3,Bryce McGuire,Wyatt Russell,Kerry Condon,Amélie Hoeferle,NaN,Night Swim
4,Jeffrey Darling,Jacob Elordi,Zachary Quinto,Patrick J. Adams,Thriller Crime Drama,He Went That Way


In [ ]:
new_df24["actor_2_name"]=new_df24["actor_2_name"].replace(np.nan,"unknown")

In [ ]:
new_df24["actor_3_name"]=new_df24["actor_3_name"].replace(np.nan,"unknown")

In [ ]:
new_df24["movie_title"]=new_df24["movie_title"].str.lower()

In [ ]:
new_df24["comb"]=new_df24["actor_1_name"].astype(str)+''+new_df24["actor_2_name"].astype(str)+""+new_df24["actor_3_name"].astype(str)+""+new_df24["director_name"].astype(str)+""+new_df24["genres"]

In [ ]:
new_df24.tail()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
490,James Mangold,Timothée Chalamet,Edward Norton,Elle Fanning,Drama Music,A Complete Unknown,Timothée Chalamet Edward Norton Elle FanningJ...
491,Rachel Morrison,Ryan Destiny,Brian Tyree Henry,Judy Greer,Drama History,The Fire Inside,Ryan Destiny Brian Tyree Henry Judy GreerRach...
492,Halina Reijn,Nicole Kidman,Harris Dickinson,Sophie Wilde,Drama Romance,Babygirl,Nicole Kidman Harris Dickinson Sophie WildeHa...
493,"Michael Schwartz, Tyler Nilson (directors/scre...",Héctor Medina,Adria Arjona,Eros de la Puente,Drama,Los Frikis,Héctor Medina Adria Arjona Eros de la PuenteM...
494,Matthew John Lawrence,Molly Brown,Jeffrey Dean Morgan,Billy Burke,Horror Comedy,Bloody Axe Wound,Molly Brown Jeffrey Dean Morgan Billy BurkeMa...


In [ ]:
my_df=pd.concat([new_df_21,new_df22,new_df23,new_df24],ignore_index=True)

my_df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Roseanne Liang,Chloë Grace Moretz,Taylor John Smith,13.0,Horror Action War,shadow in the cloud,Chloë Grace Moretz Taylor John Smith 13.0 Ro...
1,Anthony Leone,Amy Cay,Brian Patrick Butler,19.0,Adventure Family TV Movie Western,hacksaw,Amy Cay Brian Patrick Butler 19.0 Anthony Le...
2,Yaniv Raz,Lucas Jade Zumann,Taylor Russell,13.0,NaN,dr. bird's advice for sad poets,Lucas Jade Zumann Taylor Russell 13.0 Yaniv ...
3,Ramin Bahrani,Adarsh Gourav,Rajkummar Rao,22.0,Drama,the white tiger,Adarsh Gourav Rajkummar Rao 22.0 Ramin Bahra...
4,Doug Liman,Anne Hathaway,Chiwetel Ejiofor,17.0,Comedy Crime Romance,locked down,Anne Hathaway Chiwetel Ejiofor 17.0 Doug Lim...


In [ ]:
my_df.count()

director_name    1530
actor_1_name     1530
actor_2_name     1530
actor_3_name     1530
genres           1126
movie_title      1524
comb             1300
dtype: int64

In [ ]:
old_df=pd.read_csv("../datasets/final_data.csv")
old_df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...


In [ ]:
final_df =pd.concat([old_df,my_df],ignore_index=True)
final_df

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...
...,...,...,...,...,...,...,...
7388,James Mangold,Timothée Chalamet,Edward Norton,Elle Fanning,Drama Music,A Complete Unknown,Timothée Chalamet Edward Norton Elle FanningJ...
7389,Rachel Morrison,Ryan Destiny,Brian Tyree Henry,Judy Greer,Drama History,The Fire Inside,Ryan Destiny Brian Tyree Henry Judy GreerRach...
7390,Halina Reijn,Nicole Kidman,Harris Dickinson,Sophie Wilde,Drama Romance,Babygirl,Nicole Kidman Harris Dickinson Sophie WildeHa...
7391,"Michael Schwartz, Tyler Nilson (directors/scre...",Héctor Medina,Adria Arjona,Eros de la Puente,Drama,Los Frikis,Héctor Medina Adria Arjona Eros de la PuenteM...


In [ ]:
final_df.isna().sum()

director_name      0
actor_1_name       0
actor_2_name       0
actor_3_name       0
genres           404
movie_title        6
comb             230
dtype: int64

In [ ]:
final_df['genres'] = final_df['genres'].fillna('unknown')
final_df['movie_title'] = final_df['movie_title'].fillna('unknown')
final_df['comb'] = final_df['comb'].fillna('')


In [ ]:
final_df.isna().sum()

director_name    0
actor_1_name     0
actor_2_name     0
actor_3_name     0
genres           0
movie_title      0
comb             0
dtype: int64

In [15]:
import pandas as pd

In [ ]:
final_df.to_csv('../datasets/updated_final_data.csv',index=False)

In [16]:
new_df=pd.read_csv("../datasets/updated_final_data.csv")

In [17]:
old_df=pd.read_csv("../datasets/main_data.csv")
old_df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...


In [19]:
final_df =pd.concat([old_df,new_df],ignore_index=True)
final_df

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...
...,...,...,...,...,...,...,...
13499,James Mangold,Timothée Chalamet,Edward Norton,Elle Fanning,Drama Music,A Complete Unknown,Timothée Chalamet Edward Norton Elle FanningJ...
13500,Rachel Morrison,Ryan Destiny,Brian Tyree Henry,Judy Greer,Drama History,The Fire Inside,Ryan Destiny Brian Tyree Henry Judy GreerRach...
13501,Halina Reijn,Nicole Kidman,Harris Dickinson,Sophie Wilde,Drama Romance,Babygirl,Nicole Kidman Harris Dickinson Sophie WildeHa...
13502,"Michael Schwartz, Tyler Nilson (directors/scre...",Héctor Medina,Adria Arjona,Eros de la Puente,Drama,Los Frikis,Héctor Medina Adria Arjona Eros de la PuenteM...


In [21]:
final_df.shape

(13504, 7)

In [22]:
final_df['genres'] = final_df['genres'].fillna('unknown')
final_df['movie_title'] = final_df['movie_title'].fillna('unknown')
final_df['comb'] = final_df['comb'].fillna('')


In [23]:
final_df.isnull().sum()

director_name    6
actor_1_name     6
actor_2_name     0
actor_3_name     0
genres           0
movie_title      0
comb             0
dtype: int64

In [24]:
final_df.to_csv('../datasets/updated_data.csv',index=False)